# Tuần 1 — Retrieval Baseline (VLSP2025 DRiLL)

Mục tiêu của notebook này (theo kế hoạch tuần trong `SELF_RAG_SEMINAR_PROJECT_GUIDELINE.md`, mục §29 "Tuần 1"):

1. Load corpus luật (`final/dataset/VLQA/legal_corpus.json`, 2157 văn bản / 59636 điều luật, `aid` là ID toàn cục duy nhất).
2. Chunk các điều luật quá dài.
3. Encode chunk bằng embedding model tiếng Việt, build FAISS index, lưu xuống Drive để không phải encode lại mỗi lần mở Colab.
4. Tách một dev set cố định từ `train.json` (2190 câu hỏi có gán nhãn `relevant_laws` + `answer` thật — dữ liệu thật của shared task VLSP2025 DRiLL, không phải tự bịa).
5. Đánh giá retriever bằng Recall@k / Precision@k / MRR — số liệu định lượng thật để đưa vào báo cáo, thay vì chấm tay.

Đây là baseline **dense-only**. Baseline generator (Tuần 2) và 4 module reflection Self-RAG-inspired (Tuần 3) sẽ là notebook riêng, dùng lại index/artifacts được lưu ở đây.

## 0. Cấu hình môi trường (Colab hoặc local)

- Trên Colab: mount Google Drive **chỉ để lưu dữ liệu/artifact**, không `git clone` vào Drive (Google Drive FUSE hay lỗi `invalid index-pack output` với thao tác ghi pack-object của git — xem PIPELINE.md §6.4). Tạo một thư mục thường trên Drive, ví dụ `MyDrive/NLP-CS2308.CH203-data/`, rồi upload 5 file json trong `final/dataset/VLQA/` vào `.../VLQA/` bên trong đó (một lần duy nhất). Sửa biến `DRIVE_DATA_ROOT` bên dưới nếu bạn đặt tên khác.
- Mở notebook này trực tiếp từ GitHub trong Colab (File > Open notebook > GitHub, hoặc link `https://colab.research.google.com/github/HiimDManh/NLP-CS2308.CH203/blob/main/final/notebooks/01_retrieval_baseline.ipynb`) — không cần `git clone` repo.
- Chạy local: notebook nằm ở `final/notebooks/`, nên lên 2 cấp là gốc repo — không cần sửa gì.

In [ ]:
!pip install -q sentence-transformers faiss-cpu

In [ ]:
import os

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # KHONG git clone vao Drive: Google Drive FUSE khong tuong thich voi thao tac ghi
    # pack-object cua git (loi "invalid index-pack output"). Drive chi dung de luu
    # du lieu/artifact thuan tuy; notebook duoc mo truc tiep tu GitHub trong Colab.
    DRIVE_DATA_ROOT = "/content/drive/MyDrive/NLP-CS2308.CH203-data"  # sua neu ban dat ten khac
    DATA_DIR = os.path.join(DRIVE_DATA_ROOT, "VLQA")
    ARTIFACT_DIR = os.path.join(DRIVE_DATA_ROOT, "artifacts")
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    DATA_DIR = os.path.join(REPO_DIR, "final", "dataset", "VLQA")
    ARTIFACT_DIR = os.path.join(REPO_DIR, "final", "artifacts")

os.makedirs(ARTIFACT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| exists:", os.path.isdir(DATA_DIR))
print("ARTIFACT_DIR:", ARTIFACT_DIR)

## 1. Load corpus & xây dựng lookup `aid -> (law_id, text)`

`aid` đã được verify là ID duy nhất trên toàn corpus (0–59635, không trùng giữa các văn bản), nên dùng thẳng làm khóa toàn cục — không cần ghép với `law_id`.

In [ ]:
import json
import numpy as np

with open(os.path.join(DATA_DIR, "legal_corpus.json"), encoding="utf-8") as f:
    corpus = json.load(f)

aid_to_article = {}
for doc in corpus:
    for art in doc["content"]:
        aid_to_article[art["aid"]] = {"law_id": doc["law_id"], "text": art["content_Article"]}

print(f"So van ban luat: {len(corpus)}")
print(f"So dieu luat (aid) toan cuc: {len(aid_to_article)}")

lengths = [len(a["text"]) for a in aid_to_article.values()]
print(f"Do dai dieu luat (ky tu): min={min(lengths)}, median={int(np.median(lengths))}, "
      f"p95={int(np.percentile(lengths, 95))}, max={max(lengths)}")

## 2. Chunk các điều luật quá dài

Chiến lược: ưu tiên tách theo đoạn (`\n\n`); nếu một đoạn vẫn quá dài thì cắt cứng theo cửa sổ trượt có overlap. `max_chars=800` được chọn vì đa số điều luật (median ở trên) ngắn hơn ngưỡng này — chỉ các điều dài bất thường (đuôi phân phối, ví dụ điều liệt kê nhiều tiêu chuẩn/nhiệm vụ) mới bị tách.

In [ ]:
import re


def split_article(text, max_chars=800, overlap=150):
    if len(text) <= max_chars:
        return [text]

    paragraphs = re.split(r"\n\s*\n", text.strip())
    chunks, buf = [], ""
    for para in paragraphs:
        candidate = f"{buf}\n\n{para}".strip() if buf else para
        if len(candidate) <= max_chars:
            buf = candidate
            continue
        if buf:
            chunks.append(buf)
        if len(para) <= max_chars:
            buf = para
        else:
            start = 0
            while start < len(para):
                end = start + max_chars
                chunks.append(para[start:end])
                start = end - overlap
            buf = ""
    if buf:
        chunks.append(buf)
    return chunks


chunks = []
for aid, art in aid_to_article.items():
    for i, piece in enumerate(split_article(art["text"])):
        chunks.append({"chunk_id": f"{aid}_{i}", "aid": aid, "law_id": art["law_id"], "text": piece})

print(f"So dieu luat goc: {len(aid_to_article)}")
print(f"So chunk sau khi tach: {len(chunks)}")

## 3. Embedding + FAISS index

Model mặc định: `bkai-foundation-models/vietnamese-bi-encoder` (song song có thể thử `BAAI/bge-m3` nếu muốn multilingual robustness hơn). Bước encode ~60k chunk chỉ cần chạy **một lần** — index và metadata được lưu vào `ARTIFACT_DIR` (trên Drive nếu đang ở Colab), lần chạy sau sẽ tự load lại thay vì encode lại từ đầu.

In [ ]:
import torch
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
INDEX_PATH = os.path.join(ARTIFACT_DIR, "chunks.faiss")
CHUNKS_META_PATH = os.path.join(ARTIFACT_DIR, "chunks_meta.json")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print("Device:", device)

if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_META_PATH):
    index = faiss.read_index(INDEX_PATH)
    with open(CHUNKS_META_PATH, encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"Da load index co san: {index.ntotal} chunk")
else:
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(
        texts, batch_size=128, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)
    with open(CHUNKS_META_PATH, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)
    print(f"Da build va luu index: {index.ntotal} chunk, dim={embeddings.shape[1]}")

## 4. Load `train.json` & tách dev set cố định

`public_test.json`/`private_test.json` có nhãn bị ẩn (dùng cho leaderboard chính thức) nên **không dùng để tự đánh giá cục bộ được**. Dùng `train.json` (2190 câu, có `relevant_laws` + `answer` thật) làm nguồn dev set — tách cố định 250 câu, seed 42, lưu danh sách `qid` xuống artifact để mọi lần chạy sau (và mọi notebook Tuần 2/3 sau này) dùng đúng cùng một dev set, đảm bảo so sánh công bằng giữa các baseline.

In [ ]:
import random

with open(os.path.join(DATA_DIR, "train.json"), encoding="utf-8") as f:
    train_full = json.load(f)

missing = [ex["qid"] for ex in train_full if any(aid not in aid_to_article for aid in ex["relevant_laws"])]
print(f"So cau hoi co relevant_laws tro toi aid khong ton tai trong corpus: {len(missing)}")

DEV_SIZE = 250
SPLIT_PATH = os.path.join(ARTIFACT_DIR, "dev_split_qids.json")

if os.path.exists(SPLIT_PATH):
    with open(SPLIT_PATH, encoding="utf-8") as f:
        dev_qids = set(json.load(f))
else:
    rng = random.Random(42)
    shuffled = train_full[:]
    rng.shuffle(shuffled)
    dev_qids = {ex["qid"] for ex in shuffled[:DEV_SIZE]}
    with open(SPLIT_PATH, "w", encoding="utf-8") as f:
        json.dump(sorted(dev_qids), f)

dev_set = [ex for ex in train_full if ex["qid"] in dev_qids]
train_pool = [ex for ex in train_full if ex["qid"] not in dev_qids]
print(f"Dev set: {len(dev_set)} cau | Train pool (vi du minh hoa / few-shot): {len(train_pool)} cau")

## 5. Hàm retrieve: câu hỏi → danh sách `aid` xếp hạng

Search top `top_chunks` theo cosine similarity (FAISS `IndexFlatIP` trên vector đã normalize = cosine), sau đó dedup theo `aid` (nhiều chunk có thể cùng thuộc một điều luật) để ra danh sách điều luật xếp hạng — đúng đơn vị mà `relevant_laws` dùng để chấm.

In [ ]:
def retrieve(query, top_chunks=50, max_k=20):
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    _, idxs = index.search(q_emb, top_chunks)

    ranked_aids, seen = [], set()
    for idx in idxs[0]:
        aid = chunks[idx]["aid"]
        if aid not in seen:
            seen.add(aid)
            ranked_aids.append(aid)
        if len(ranked_aids) >= max_k:
            break
    return ranked_aids

## 6. Đánh giá Recall@k / Precision@k / MRR trên dev set

In [ ]:
def recall_at_k(retrieved, gold, k):
    if not gold:
        return None
    hit = len(set(retrieved[:k]) & set(gold))
    return hit / len(gold)


def precision_at_k(retrieved, gold, k):
    hit = len(set(retrieved[:k]) & set(gold))
    return hit / k


def reciprocal_rank(retrieved, gold):
    for rank, aid in enumerate(retrieved, start=1):
        if aid in gold:
            return 1.0 / rank
    return 0.0


K_VALUES = [1, 3, 5, 10, 20]
per_question = []
for ex in dev_set:
    ranked = retrieve(ex["question"], top_chunks=50, max_k=max(K_VALUES))
    gold = set(ex["relevant_laws"])
    row = {"qid": ex["qid"]}
    for k in K_VALUES:
        row[f"recall@{k}"] = recall_at_k(ranked, gold, k)
        row[f"precision@{k}"] = precision_at_k(ranked, gold, k)
    row["mrr"] = reciprocal_rank(ranked, gold)
    per_question.append(row)

print(f"Da danh gia {len(per_question)} cau hoi trong dev set")

In [ ]:
import pandas as pd

per_question_df = pd.DataFrame(per_question)
summary = pd.DataFrame({
    "Recall@k": [per_question_df[f"recall@{k}"].mean() for k in K_VALUES],
    "Precision@k": [per_question_df[f"precision@{k}"].mean() for k in K_VALUES],
}, index=[f"k={k}" for k in K_VALUES])
print(f"MRR trung binh (rank tren top-{max(K_VALUES)}): {per_question_df['mrr'].mean():.4f}")
summary

In [ ]:
METRICS_PATH = os.path.join(ARTIFACT_DIR, "retrieval_baseline_metrics.csv")
per_question_df.to_csv(METRICS_PATH, index=False)
print(f"Da luu ket qua chi tiet tung cau hoi: {METRICS_PATH}")

## 7. Soi vài case cụ thể (định tính)

Dùng cho phần "case dùng/sai" trong báo cáo (guideline §28, mục Results).

In [ ]:
sample_qids = random.Random(0).sample([ex["qid"] for ex in dev_set], 3)
for qid in sample_qids:
    ex = next(e for e in dev_set if e["qid"] == qid)
    ranked = retrieve(ex["question"], top_chunks=50, max_k=5)
    gold = set(ex["relevant_laws"])
    print(f"qid={qid}")
    print("Cau hoi:", ex["question"])
    print("Gold aid:", sorted(gold))
    print("Top-5 retrieved aid:", ranked)
    print("Hit:", sorted(gold & set(ranked)))
    print("-" * 80)

## 8. Bước tiếp theo

- **Tuần 2**: baseline generator — prompt LLM (API free tier) với câu hỏi + top-k passage từ `retrieve()`, sinh câu trả lời, so sánh định tính với `answer` gold.
- **Tuần 3**: 4 module reflection Self-RAG-inspired (Retrieve-decision, ISREL, ISSUP, ISUSE) — dùng cùng `dev_set`/`train_pool` và `chunks.faiss` đã build ở đây.
- **Stretch (nếu còn thời gian)**: thêm BM25 (`pyvi`/`underthesea` tokenizer) làm hybrid retriever, so Recall@k dense-only vs. hybrid — văn bản luật có mã số/điều khoản chính xác mà dense hay bỏ lỡ.
- **Khi hệ thống ổn định**: định dạng lại output cho `public_test.json`/`private_test.json` để nộp leaderboard VLSP2025 DRiLL (phương án (b) đã bàn — làm sau khi baseline + Self-RAG-inspired chạy ổn).